In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
import json, joblib, numpy as np, pandas as pd

PATH = "/content/drive/MyDrive/nonprofit_risk_model.parquet"
df = pd.read_parquet(PATH)

In [ ]:
#Feature Engineering and Data Cleaning
# ---------------------------
# 1️⃣ Remove impossible leverage values
# ---------------------------
df = df[
    (df["leverage_ratio"] >= 0) &
    (df["leverage_ratio"] <= 5)
]

# ---------------------------
# 2️⃣ Remove tiny orgs that distort ratios
# ---------------------------
df = df[df["total_revenue"] > 10000]

# ---------------------------
# 3️⃣ Sort for proper lag operations
# ---------------------------
df = df.sort_values(["ein", "taxyear"])

# ---------------------------
# 4️⃣ Create log size controls
# ---------------------------
df["log_revenue"] = np.log1p(df["total_revenue"])
df["log_assets"] = np.log1p(df["total_assets_end"])

# ---------------------------
# 5️⃣ Operating margin (stable version)
# ---------------------------
df["operating_margin"] = df["surplus"] / (df["total_revenue"] + 1)

# ---------------------------
# 6️⃣ Replace pct changes with log changes (stable)
# ---------------------------
df["rev_log_change"] = (
    np.log1p(df["total_revenue"]) -
    np.log1p(df.groupby("ein")["total_revenue"].shift(1))
)

df["assets_log_change"] = (
    np.log1p(df["total_assets_end"]) -
    np.log1p(df.groupby("ein")["total_assets_end"].shift(1))
)

# Drop rows where lag created nulls
df = df.dropna(subset=["rev_log_change", "assets_log_change"])

# ---------------------------
# 7️⃣ Clip extreme tails (1st–99th percentile)
# ---------------------------
for col in [
    "operating_margin",
    "expense_ratio",
    "rev_log_change",
    "assets_log_change"
]:
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    df[col] = df[col].clip(lower, upper)

# ---------------------------
# 8️⃣ Final feature list
# ---------------------------
features = [
    "log_revenue",
    "log_assets",
    "operating_margin",
    "expense_ratio",
    "leverage_ratio",
    "rev_log_change",
    "assets_log_change"
]

# **Modelling**

In [ ]:
# Time split
train = df[df["taxyear"] <= 2018]
test  = df[df["taxyear"] >= 2022]

X_train = train[features]
y_train = train["distress_next_year"]

X_test = test[features]
y_test = test["distress_next_year"]

## **Logistic Regression**

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

probs = model.predict_proba(X_test_scaled)[:, 1]

print("Test AUC:", roc_auc_score(y_test, probs))

# Top decile precision
test["risk_score"] = probs
threshold = np.percentile(probs, 90)
top_decile = test[test["risk_score"] >= threshold]
precision_top10 = top_decile["distress_next_year"].mean()

print("Base distress rate:", y_test.mean())
print("Precision at Top 10%:", precision_top10)

Test AUC: 0.649763536258849
Base distress rate: 0.31795275439387977
Precision at Top 10%: 0.5080949638446279


/tmp/ipykernel_4236/1315501236.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test["risk_score"] = probs


## **XGBoost Classifier**

In [ ]:
xgb = XGBClassifier(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="auc",
    n_jobs=-1,
    random_state=42
)

xgb.fit(X_train, y_train)
xgb_probs = xgb.predict_proba(X_test)[:, 1]

print("XGBoost Test AUC:", roc_auc_score(y_test, xgb_probs))
print("Base distress rate:", y_test.mean())
# Top decile precision
thr = np.percentile(xgb_probs, 90)
print("XGBoost Top 10% precision:", (y_test[xgb_probs >= thr].mean()))

XGBoost Test AUC: 0.6897032435776993
Base distress rate: 0.31795275439387977
XGBoost Top 10% precision: 0.5795098227426889


In [ ]:
xgb.feature_importances_

array([0.05388974, 0.11542248, 0.24137656, 0.38532752, 0.04856998,
       0.05000215, 0.10541158], dtype=float32)

# **Exports**

## **Tableau Exports**

In [ ]:
import os

project_path = "/content/drive/MyDrive/NFSRI_Project_v1"
os.makedirs(project_path, exist_ok=True)

project_path

'/content/drive/MyDrive/NFSRI_Project_v1'

In [ ]:
# Nonprofit Financial Sustainability Risk Index (0–100 scale)
X_all = df[features]
df = df.copy()
df["risk_score"] = xgb.predict_proba(X_all)[:, 1]
df["risk_index"] = (df["risk_score"] * 100).round(1)

# Deciles across entire scored dataset
df["risk_decile"] = pd.qcut(df["risk_score"], 10, labels=[f"D{i}" for i in range(1, 11)])

# Optional: simple tiers funders understand
df["risk_tier"] = pd.qcut(
    df["risk_score"], 5,
    labels=["Very Low", "Low", "Moderate", "High", "Very High"]
)

tableau_cols = [
    "ein", "taxyear",
    "risk_score", "risk_index",
    "risk_decile", "risk_tier",
    "distress_next_year",
    # include a few context fields if you still have them
    "total_revenue", "total_assets_end", "surplus"
]

# Only keep columns that exist (in case some aren’t present)
tableau_cols = [c for c in tableau_cols if c in df.columns]

out = df[tableau_cols].copy()

out.to_csv(f"{project_path}/nfsri_tableau_v1.csv", index=False)
out.to_parquet(f"{project_path}/nfsri_tableau_v1.parquet", index=False)
df.to_parquet(f"{project_path}/nfsri_modeling_dataset_v1.parquet", index=False)
print("Saved:", out.shape)

Saved: (2210667, 10)


## **Freezing Best Model**

In [ ]:
# 1) Save model
joblib.dump(xgb, f"{project_path}/nfsri_xgb_v1.pkl")

# 2) Save config (critical for reproducibility)
config = {
    "model": "XGBoost",
    "version": "v1.0",
    "train_years": "2007-2018",
    "test_years": "2022-2024",
    "features": features,
    "target": "distress_next_year",
    "label_definition": "distress_next_year = 1 if (surplus(t+1) < 0) AND (assets_log_change(t+1) < 0)",
    "data_filters": {
        "total_revenue_min": 10000,
        "leverage_ratio_range": [0, 5],
        "clip_percentiles": [0.01, 0.99]
    },
    "metrics": {
        "xgb_test_auc": 0.6897032435776993,
        "xgb_top10_precision": 0.5795098227426889
    }
}

with open(f"{project_path}/nfsri_model_config_v1.json", "w") as f:
    json.dump(config, f, indent=2)

## **Making Dataset for Actionable Layer In Tableau**

In [ ]:
import pandas as pd
import numpy as np
import os

# ----------------------------
# 0. LOAD DATA
# ----------------------------
# main dataset (already in memory as df)
df_action = df.copy()

# names / sector file
p = "/content/drive/MyDrive/ntee_sector_and_name.csv"
df_names = pd.read_csv(p)

# ----------------------------
# 1. STANDARDIZE EIN FORMAT
# ----------------------------
# rename if needed
if "ein_raw" in df_names.columns:
    df_names = df_names.rename(columns={"ein_raw": "ein"})
if "EIN" in df_names.columns:
    df_names = df_names.rename(columns={"EIN": "ein"})

# ensure string + 9 digits
df_action["ein"] = df_action["ein"].astype(str).str.zfill(9)
df_names["ein"] = df_names["ein"].astype(str).str.zfill(9)

# ----------------------------
# 2. CLEAN NAMES DATA
# ----------------------------
# keep only useful columns (adjust if needed)
keep_cols = [c for c in [
    "ein",
    "org_name",
    "sector_raw",
    "sector_group"
] if c in df_names.columns]

df_names_clean = df_names[keep_cols].copy()

# unify org name column
if "name" in df_names_clean.columns and "org_name" not in df_names_clean.columns:
    df_names_clean = df_names_clean.rename(columns={"name": "org_name"})

# drop duplicates
df_names_clean = df_names_clean.drop_duplicates(subset=["ein"])

# ----------------------------
# 3. MERGE INTO MAIN DATA
# ----------------------------
df_action = df_action.merge(
    df_names_clean,
    on="ein",
    how="left"
)

# fill missing values
df_action["org_name"] = df_action["org_name"].fillna("Unknown Organization")
if "sector_group" in df_action.columns:
    df_action["sector_group"] = df_action["sector_group"].fillna("Unknown")

# clean org names (optional)
df_action["org_name"] = (
    df_action["org_name"]
    .str.title()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# ----------------------------
# 4. CREATE REVENUE TIER
# ----------------------------
df_action["revenue_tier"] = pd.cut(
    df_action["total_revenue"],
    bins=[-np.inf, 100000, 1000000, 10000000, np.inf],
    labels=["<$100K", "$100K–$1M", "$1M–$10M", "$10M+"]
)

# ----------------------------
# 5. CREATE DRIVER FLAGS
# ----------------------------
df_action["flag_high_expense_ratio"] = (df_action["expense_ratio"] > 0.90).astype(int)
df_action["flag_low_margin"] = (df_action["operating_margin"] < 0.05).astype(int)
df_action["flag_declining_assets"] = (df_action["assets_log_change"] < 0).astype(int)
df_action["flag_declining_revenue"] = (df_action["rev_log_change"] < 0).astype(int)
df_action["flag_small_size"] = (df_action["revenue_tier"] == "<$100K").astype(int)
df_action["flag_high_leverage"] = (df_action["leverage_ratio"] > 0.70).astype(int)

# ----------------------------
# 6. TOTAL RISK FLAGS
# ----------------------------
flag_cols = [
    "flag_high_expense_ratio",
    "flag_low_margin",
    "flag_declining_assets",
    "flag_declining_revenue",
    "flag_small_size",
    "flag_high_leverage"
]

df_action["num_risk_flags"] = df_action[flag_cols].sum(axis=1)

# ----------------------------
# 7. PRIMARY DRIVER
# ----------------------------
def get_primary_driver(row):
    if row["flag_high_expense_ratio"]:
        return "High Expense Ratio"
    elif row["flag_low_margin"]:
        return "Low / Negative Margin"
    elif row["flag_declining_assets"]:
        return "Declining Assets"
    elif row["flag_declining_revenue"]:
        return "Declining Revenue"
    elif row["flag_high_leverage"]:
        return "High Leverage"
    elif row["flag_small_size"]:
        return "Small Organization"
    else:
        return "No Primary Flag"

df_action["primary_risk_driver"] = df_action.apply(get_primary_driver, axis=1)

# ----------------------------
# 8. KEEP LATEST YEAR PER EIN
# ----------------------------
df_latest = (
    df_action
    .sort_values(["ein", "taxyear"])
    .groupby("ein", as_index=False)
    .tail(1)
)

# ----------------------------
# 9. SELECT FINAL TABLEAU COLUMNS
# ----------------------------
tableau_cols = [c for c in [
    "ein",
    "org_name",
    "taxyear",
    "risk_score",
    "risk_index",
    "risk_decile",
    "risk_tier",
    "total_revenue",
    "total_assets_end",
    "surplus",
    "expense_ratio",
    "operating_margin",
    "leverage_ratio",
    "rev_log_change",
    "assets_log_change",
    "revenue_tier",
    "sector_raw",
    "sector_group",
    "num_risk_flags",
    "primary_risk_driver",
    "flag_high_expense_ratio",
    "flag_low_margin",
    "flag_declining_assets",
    "flag_declining_revenue",
    "flag_small_size",
    "flag_high_leverage"
] if c in df_latest.columns]

df_latest_tableau = df_latest[tableau_cols].copy()

# ----------------------------
# 10. SAVE TO DRIVE
# ----------------------------
project_path = "/content/drive/MyDrive/NFSRI_Project_v1"
os.makedirs(project_path, exist_ok=True)

df_latest_tableau.to_csv(
    f"{project_path}/nfsri_latest_org_risk_profiles_v1.csv",
    index=False
)

print("✅ Saved file:")
print(df_latest_tableau.shape)
print(df_latest_tableau.head())

✅ Saved file:
(335438, 26)
          ein                                           org_name  taxyear  \
1   010018922                                    American Legion     2022   
4   010018923       American Legion Post 0019 Thomas W Cole Post     2021   
9   010018927          American Legion Post 0005 Bourque Lanigan     2020   
20  010018930                                    American Legion     2022   
30  010019705  Ancient Free & Accepted Masons Of Maine Grand ...     2022   

    risk_score  risk_index risk_decile  risk_tier  total_revenue  \
1     0.307398   30.700001          D6   Moderate       231785.0   
4     0.319534   32.000000          D6   Moderate       260386.0   
9     0.531770   53.200001         D10  Very High        14145.0   
20    0.460590   46.099998          D9  Very High       887494.0   
30    0.181872   18.200001          D3        Low       384203.0   

    total_assets_end  surplus  ...  sector_raw            sector_group  \
1           271690.0  29238